# Phase 5 — Extraction d'entités et exploitation du layout

**Objectif** : passer de la reconnaissance de texte brute (Phase 3/4) à une **information structurée** (paires clé-valeur), en exploitant les coordonnées spatiales des mots.

Deux approches comparées (cf. taxonomie du cahier des charges) :
1. **Règles + OCR** : détection de motifs génériques (dates, montants, références...) par expressions régulières sur le texte OCR brut.
2. **OCR + layout** : regroupement des mots en lignes via leurs coordonnées, puis association label → valeur par proximité spatiale.

La 3e approche possible, **layout-aware (LayoutLM/LayoutLMv3)**, est documentée mais non implémentée ici (nécessite un GPU — cf. note de synthèse en fin de notebook et section "Risques" du cahier des charges).

**Vérité terrain** : FUNSD fournit directement des relations question→réponse annotées (champ `linking`), qu'on utilise comme référence pour évaluer nos extractions.

In [2]:
import sys
import json
from pathlib import Path

sys.path.append("..")
from src.ocr_engine import run_ocr
from src.extraction import (
    extract_entities_by_rules,
    extract_key_value_pairs_spatial,
    ground_truth_pairs_from_annotation,
    evaluate_extraction,
)

import pandas as pd
from tqdm import tqdm

ROOT = Path("..")
RAW_DIR = ROOT / "data" / "raw"
DEGRADED_DIR = ROOT / "data" / "degraded"
RESULTS_DIR = ROOT / "results"
(RESULTS_DIR / "tables").mkdir(parents=True, exist_ok=True)

manifest = json.loads((RAW_DIR / "manifest.json").read_text(encoding="utf-8"))
print(f"Documents disponibles : {len(manifest)}")

Documents disponibles : 149


## 1. Démonstration sur un document exemple

On regarde en détail ce que chaque approche extrait, avant de lancer l'évaluation complète.

In [3]:
sample_entry = manifest[0]
img_path = RAW_DIR / "images" / sample_entry["image"]
ann_path = RAW_DIR / "annotations" / sample_entry["annotation"]

ocr_result = run_ocr(img_path)
print(f"--- Document : {sample_entry['image']} ({len(ocr_result['words'])} mots OCR) ---\n")

print(">>> Approche 1 : regles (motifs generiques dans le texte)")
rule_entities = extract_entities_by_rules(ocr_result["text"])
for e in rule_entities[:10]:
    print(" -", e)

print()
print(">>> Approche 2 : OCR + layout (association spatiale label -> valeur)")
spatial_pairs = extract_key_value_pairs_spatial(ocr_result["words"])
for p in spatial_pairs[:10]:
    print(" -", p)

print()
print(">>> Verite terrain FUNSD (linking question -> answer)")
gt_pairs = ground_truth_pairs_from_annotation(ann_path)
for p in gt_pairs[:10]:
    print(" -", p)

--- Document : 0000971160.png (127 mots OCR) ---

>>> Approche 1 : regles (motifs generiques dans le texte)
 - {'type': 'date', 'valeur': '9/3/92', 'position_char': 108}

>>> Approche 2 : OCR + layout (association spatiale label -> valeur)
 - {'label': 'Ext.:', 'valeur': 'M. Hamann, P, Harper,', 'methode': 'meme_ligne'}
 - {'label': 'Date:', 'valeur': '__9/3/92', 'methode': 'meme_ligne'}
 - {'label': 'Supervisor/Manager:', 'valeur': 'J. S. Wigand RED', 'methode': 'meme_ligne'}
 - {'label': 'Group:', 'valeur': '_Licensee', 'methode': 'meme_ligne'}
 - {'label': 'Suggestion:', 'valeur': 'Discontinue coal retention analyses', 'methode': 'meme_ligne'}
 - {'label': '(Note:', 'valeur': 'Coal Retention testing is', 'methode': 'meme_ligne'}
 - {'label': 'Solution(s):', 'valeur': 'Delete coal retention from', 'methode': 'meme_ligne'}
 - {'label': 'Manager', 'valeur': 'Connents: Manager, please contact', 'methode': 'meme_ligne'}
 - {'label': 'Connents:', 'valeur': 'Manager, please contact suggest

## 2. Évaluation de l'approche spatiale sur l'ensemble des documents

Comparaison des paires extraites (approche 2) à la vérité terrain FUNSD, avec les métriques exact match / partial match demandées par le cahier des charges.

In [4]:
results_original = []

for entry in tqdm(manifest, desc="Extraction sur documents originaux"):
    img_path = RAW_DIR / "images" / entry["image"]
    ann_path = RAW_DIR / "annotations" / entry["annotation"]

    ocr_result = run_ocr(img_path)
    pred_pairs = extract_key_value_pairs_spatial(ocr_result["words"])
    gt_pairs = ground_truth_pairs_from_annotation(ann_path)
    metrics = evaluate_extraction(pred_pairs, gt_pairs)

    results_original.append({"document": entry["image"], "condition": "original", **metrics})

df_original = pd.DataFrame(results_original)
df_original.to_csv(RESULTS_DIR / "tables" / "phase5_extraction_original.csv", index=False)
df_original[["precision_partial","recall_partial","f1_partial","taux_labels_retrouves"]].describe().round(3)

Extraction sur documents originaux: 100%|██████████| 149/149 [06:20<00:00,  2.55s/it]


,precision_partial,recall_partial,f1_partial,taux_labels_retrouves
count,149.000,149.000,149.000,149.000
mean,0.134,0.084,0.090,0.142
std,0.186,0.131,0.124,0.168
min,0.000,0.000,0.000,0.000
25%,0.000,0.000,0.000,0.000
50%,0.071,0.033,0.045,0.095
75%,0.214,0.111,0.133,0.200
max,1.000,0.667,0.615,0.750


## 3. Comparaison documents propres vs documents dégradés

On refait la même évaluation sur un échantillon de documents dégradés (Phase 2), pour répondre à la question du cahier des charges : *'l'exploitation de la mise en page améliore-t-elle la reconnaissance des entités, et comment se dégrade-t-elle avec le bruit ?'*

In [5]:
import random
random.seed(42)

manifest_degraded_path = DEGRADED_DIR / "manifest_degraded.json"
if manifest_degraded_path.exists():
    manifest_degraded = json.loads(manifest_degraded_path.read_text(encoding="utf-8"))

    # echantillon : quelques documents par niveau (fort = le plus parlant)
    sample_degraded = [e for e in manifest_degraded if e["level"] == "fort"]
    sample_degraded = random.sample(sample_degraded, min(40, len(sample_degraded)))

    results_degraded = []
    for entry in tqdm(sample_degraded, desc="Extraction sur documents degrades (niveau fort)"):
        img_path = DEGRADED_DIR / "images" / entry["image_degradee"]
        ann_path = RAW_DIR / "annotations" / entry["annotation_originale"]

        ocr_result = run_ocr(img_path)
        pred_pairs = extract_key_value_pairs_spatial(ocr_result["words"])
        gt_pairs = ground_truth_pairs_from_annotation(ann_path)
        metrics = evaluate_extraction(pred_pairs, gt_pairs)

        results_degraded.append({
            "document": entry["document_original"], "condition": "degrade",
            "degradation": entry["degradation"], **metrics,
        })

    df_degraded = pd.DataFrame(results_degraded)
    df_degraded.to_csv(RESULTS_DIR / "tables" / "phase5_extraction_degraded.csv", index=False)

    print("Comparaison ORIGINAL vs DEGRADE (niveau fort) :")
    comparison = pd.DataFrame({
        "original": df_original[["precision_partial","recall_partial","f1_partial"]].mean(),
        "degrade_fort": df_degraded[["precision_partial","recall_partial","f1_partial"]].mean(),
    }).round(3)
    print(comparison)
else:
    print("Pas de data/degraded/manifest_degraded.json trouve -- lance d'abord la Phase 2.")

Extraction sur documents degrades (niveau fort): 100%|██████████| 40/40 [01:13<00:00,  1.84s/it]

Comparaison ORIGINAL vs DEGRADE (niveau fort) :
                   original  degrade_fort
precision_partial     0.134         0.060
recall_partial        0.084         0.027
f1_partial            0.090         0.028


## 4. Sauvegarde d'un exemple de sortie structurée (JSON)

Le cahier des charges demande une sortie structurée par document — voici le format utilisé.

In [6]:
sortie_structuree = {
    "document": sample_entry["image"],
    "paires_extraites": spatial_pairs,
    "methode": "ocr_layout_spatial",
}

output_path = RESULTS_DIR / "tables" / f"phase5_extraction_exemple_{Path(sample_entry['image']).stem}.json"
output_path.write_text(json.dumps(sortie_structuree, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Exemple de sortie structuree sauvegarde : {output_path}")
print(json.dumps(sortie_structuree, ensure_ascii=False, indent=2)[:600])

Exemple de sortie structuree sauvegarde : ..\results\tables\phase5_extraction_exemple_0000971160.json
{
  "document": "0000971160.png",
  "paires_extraites": [
    {
      "label": "Ext.:",
      "valeur": "M. Hamann, P, Harper,",
      "methode": "meme_ligne"
    },
    {
      "label": "Date:",
      "valeur": "__9/3/92",
      "methode": "meme_ligne"
    },
    {
      "label": "Supervisor/Manager:",
      "valeur": "J. S. Wigand RED",
      "methode": "meme_ligne"
    },
    {
      "label": "Group:",
      "valeur": "_Licensee",
      "methode": "meme_ligne"
    },
    {
      "label": "Suggestion:",
      "valeur": "Discontinue coal retention analyses",
      "methode": "meme_ligne"
    


## 5. Note de synthèse (résultats attendus de la Phase 5)

- **Module d'extraction** : `src/extraction.py` — approche par règles (regex génériques) + approche spatiale/layout (association label → valeur par proximité de coordonnées).
- **Sortie structurée** : format JSON par document (section 4), également exportable en CSV agrégé (sections 2-3).
- **Évaluation** : exact match ET partial match (comme demandé par le cahier des charges) — l'exact match est très sévère avec un OCR imparfait, le partial match donne une vision plus réaliste de la qualité.
- **Comparaison propre vs dégradé** : la qualité d'extraction chute avec le bruit (section 3) — cohérent avec la baseline OCR de la Phase 3 (moins de texte bien reconnu = moins de paires correctement extraites).
- **Limite à mentionner dans le rapport** : notre approche spatiale est une heuristique simple (proximité géométrique), pas un modèle appris. Ses scores restent modestes par rapport à ce qu'obtiendrait un modèle layout-aware fine-tuné (LayoutLM/LayoutLMv3 atteignent typiquement un F1 bien supérieur sur FUNSD dans la littérature), mais un tel fine-tuning nécessite un GPU — cf. section "Risques et mitigation" du cahier des charges. C'est une piste explicite de travail futur (Master 2 / thèse), pas un manque de ce PFA.
- **Prochaine étape (Phase 6)** : construire un score de confiance par champ extrait et une matrice d'erreurs, pour savoir *quand* faire confiance à une extraction.